# Benchmark de modelos ocultos — Classificação tabular em big data (resposta clássica)

**Problema:** Predict customer churn (binary) on a big tabular dataset: about 5 million rows and 50 mixed numeric and high-cardinality categorical features. I need strong accuracy and training that scales. Which model would you use?

**Tipo:** LIMITE (resposta clássica; nenhum oculto cabe)

**Modelo oculto em questão:** (nenhum esperado)

## Metodologia

Concorrentes **crus** (sem ferramenta, da própria memória) em dois níveis (básico: gemini-2.5-flash, gpt-5-mini, Claude Sonnet; fronteira: gemini-2.5-pro, gpt-5, Claude Opus). A **ferramenta** busca no índice, filtra ao conjunto curado de **modelos ocultos** (data/hidden_models.json) e reordena por `stat_fit`. Pergunta: o modelo crus lembra do oculto ideal, ou esquece? gemini/openai por API; Sonnet/Opus por subagente.

## Nível básico (crus)

**gemini-2.5-flash**

I would use **LightGBM** (specifically, a Gradient Boosting Classifier).

**Why:** LightGBM offers an excellent balance of high predictive accuracy, superior training speed, and scalability for large tabular datasets compared to other tree-based methods. It efficiently handles high-cardinality categorical features directly, reducing the need for extensive preprocessing.

**Key Trade-off:** Its aggressive optimization for speed and memory, particularly its leaf-wise tree growth, can make it more susceptible to overfitting compared to more conservative boosting algorithms if regularization parameters are not carefully tuned.

**gpt-5-mini**

Model: CatBoost (CatBoostClassifier).

Why: native handling of high‑cardinality categorical features (ordered target statistics), strong out‑of‑the‑box accuracy on tabular data, and scalable multi‑thread/GPU training so you can run on 5M rows with minimal feature engineering.

Key trade‑off: typically slower and more memory‑hungry to train than LightGBM/XGBoost and tuning can take longer — you trade some training speed/resources for better handling of categorical variables and often higher final accuracy.

**Claude Sonnet** (claude-sonnet-4-6, subagente)

MODEL: XGBoost

Handles mixed numeric and high-cardinality categoricals and its histogram-based tree building scales to 5 million rows with controlled memory; optimizes log-loss directly for top tabular accuracy. Trade-off: hyperparameter tuning is non-trivial and training grows with tree count, though parallelism helps.

## Nível fronteira (crus)

**gemini-2.5-pro**

I would use **LightGBM**.

*   **Why:** It is a gradient boosting framework specifically designed for speed and efficiency on large datasets. Its key advantage is its native, optimized handling of high-cardinality categorical features, which simplifies preprocessing and often improves accuracy.

*   **Key Trade-off:** Its leaf-wise tree growth, which makes it fast, also makes it more prone to overfitting than other boosting models, requiring careful hyperparameter tuning (especially `num_leaves` and `max_depth`).

**gpt-5**

CatBoostClassifier.

Why: It natively handles high‑cardinality categorical features with ordered target statistics, delivering top accuracy on tabular data with minimal preprocessing. It scales to millions of rows with efficient CPU multithreading or GPU training.

Key trade-off: Typically slower and more memory‑hungry than LightGBM on very large datasets; you pay higher training cost for better handling of categorical features and usually higher accuracy.

**Claude Opus** (claude-opus-4-8, subagente)

MODEL: LightGBM

Built for this regime: 5M rows train fast via histogram split finding, native high-cardinality categorical handling, and GBDTs are state-of-the-art on tabular for binary churn. Trade-off: leaf-wise growth needs careful regularization/early stopping and offers less out-of-the-box interpretability (lean on SHAP).

## Ferramenta (modelos ocultos que cabem)

Filtrado ao conjunto de ocultos, reordenado por `stat_fit`. Candidatos ocultos recuperados: 2.

| # | modelo oculto | ano | fitScore | razões |
|---|---|---|---|---|
| 1 | NGBoost (probabilistic) | 2019 | +3.52 | +target binary; ~features partial |
| 2 | TabPFN | 2022 | +0.51 | +target binary; ~features partial; -n=5000000 above ceiling (~10000) |

**Oculto-alvo no top-3:** NÃO

## Análise imparcial

| Concorrente | Nível | Nomeou |
|---|---|---|
| gemini-2.5-flash | básico | LightGBM |
| gpt-5-mini | básico | CatBoost |
| Claude Sonnet | básico | XGBoost |
| gemini-2.5-pro | fronteira | LightGBM |
| gpt-5 | fronteira | CatBoost |
| Claude Opus | fronteira | LightGBM |
| **Ferramenta** | — | nenhum oculto compelativo |

**Limite.** Big tabular = booster, que os 6 crus nomeiam de cor (LightGBM / CatBoost / XGBoost). Entre os ocultos, o TabPFN é contraindicado (n > 10k) e o NGBoost é para regressão probabilística - nenhum cabe de forma compelativa. A ferramenta corretamente não força nada e concorda com a resposta clássica.

## Reprodução

In [ ]:
import bench_lib as B
case = B.case_by_name('big_data_tabular')
# cru (pago):
print(B.call_gemini(case['prompt'], B.TIERS['frontier']['gemini'])[0])
# ferramenta de ocultos (grátis):
import json; print(json.dumps(B.tool_overlooked(case), indent=2, ensure_ascii=False))